In [ ]:
import os
import sys


current = os.getcwd()
while not os.path.exists(os.path.join(current, "data")):
    parent = os.path.dirname(current)
    if parent == current:
        raise Exception("Could not find root directory.")
    current = parent

sys.path.insert(0, current)

from src.ingest import PDFParser, ContentClassifier, Tokenizer
from src.store import Chunker, ChunkStore, BM25Index

C:\Users\abeku\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import glob
import pandas as pd


data_path = os.path.join(current, "data", "raw")
output_path = os.path.join(current, "data", "processed")
os.makedirs(output_path, exist_ok=True)

pdf_files = glob.glob(os.path.join(data_path, "*.pdf"))

classifier = ContentClassifier()
tokenizer = Tokenizer('bert-base-uncased')
# We are using WordPiece tokenizer and adding semantic classification!
chunker = Chunker(tokenizer, classifier, max_tokens=150, overlap=30)
store = ChunkStore()

for pdf_file in pdf_files:
    print(f"Processing: {pdf_file}")
    
    parser = PDFParser(pdf_file)
    parsed_pages = parser.extract()
    
    chunks = chunker.chunk(parsed_pages)
    
    for chunk in chunks:
        chunk["metadata"]["source"] = {"file_name": os.path.basename(pdf_file)}
        
    store.add_chunks(chunks)

# Save JSONL securely via ChunkStore
jsonl_path = os.path.join(output_path, "chunks_v3.jsonl")
store.save_jsonl(jsonl_path)
print(f"Saved JSONL to: {jsonl_path}")

# Preview with pandas
df = pd.DataFrame(store.chunks)
metadata_df = pd.json_normalize(df["metadata"])
df = pd.concat([df.drop(columns=["metadata"]), metadata_df], axis=1)

df.head(3)

Processing: C:\Users\abeku\OneDrive\Desktop\Studybuddy\StudyBuddy-AI\data\raw\iesc102.pdf
Saved JSONL to: C:\Users\abeku\OneDrive\Desktop\Studybuddy\StudyBuddy-AI\data\processed\chunks_v3.jsonl


,id,text,chapter,content_type,token_count,page_range,section.number,section.title,source.file_name
0,f99dd6d4-ab05-4351-8690-6a8dbecd3f2b,it is widely accepted amongst the scientific c...,None,Concept,150,"[1, 1]",3.5,billion years ago,iesc102.pdf
1,15c744a9-c189-40cf-a106-c53c07c15944,unicellular. scientists from the birbal sahni ...,None,Concept,150,"[1, 1]",3.5,billion years ago,iesc102.pdf
2,f0365189-e7b8-42c1-b332-762a6d7146b3,", fish, birds or humans are made up of million...",None,Concept,150,"[1, 2]",2.1,How to Study Cells,iesc102.pdf


In [46]:

index = BM25Index(tokenizer)
index.build_index(store)

print("Searching ranking for: 'cell membrane'\n")
results = index.search("cell membrane", top_k=3)
for i, res in enumerate(results):
    print(f"Result {i+1} (Score: {res['score']:.2f}):")
    print(res['chunk']['text'][:200] + "...\n")

Searching ranking for: 'cell membrane'

Result 1 (Score: 2.69):
another and with their surroundings. these interactions occur at the cell boundary, where substances move between the cells and their external environment. even single - celled organisms exchange mate...

Result 2 (Score: 2.51):
# # m. in contrast, plant and animal cells have a well - defined nucleus and several membrane - bound organelles. such cells are called eukaryo # # tic cells ( eu means true, and karyon means nucleus ...

Result 3 (Score: 2.44):
or meiosis model with your classmates for your science project or exhibition. how did teamwork contribute to the success of the activity? did this activity change your perspective or understanding of ...

